# Lezione 4 — Buone pratiche di addestramento

Corso "Fondamenti di TensorFlow da zero" — DIH DATAlife

## Obiettivi di questa lezione

- Rilevare sovraffittamento (overfitting) e sottoadattamento (underfitting) dalle curve di addestramento e validazione
- Applicare tecniche di regolarizzazione: dropout, regolarizzazione L2 ed early stopping
- Salvare e caricare un modello addestrato e i suoi pesi per riutilizzarlo

Lavoriamo con il dataset IMDB (Internet Movie Database) di recensioni di film, già incluso in Keras, perché è lo stesso esempio che il materiale della certificazione EITC/AI/TFF (European IT Certificate / Artificial Intelligence / TensorFlow Fundamentals) usa per introdurre il sovraffittamento.

## 1. Sovraffittamento e sottoadattamento

- **Sovraffittamento:** la perdita (loss, l'errore del modello) di addestramento continua a diminuire, ma quella di validazione si stabilizza o sale. Il modello memorizza invece di generalizzare
- **Sottoadattamento:** entrambe le perdite rimangono alte. Il modello non ha capacità o addestramento sufficiente

Per vederlo con i nostri stessi occhi, andiamo ad addestrare tre modelli con diversa capacità (numero di neuroni) sullo stesso problema e confrontare le loro curve.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, regularizers
import matplotlib.pyplot as plt

print("Versión de TensorFlow:", tf.__version__)

### Prepariamo i dati

Il dataset IMDB contiene 50.000 recensioni di film etichettate come positive o negative. Ogni recensione arriva già convertita in una lista di numeri: ogni numero è l'ID di una parola (l'ID 1 è la parola più frequente, l'ID 2 la seconda più frequente, e così via).

Per poter inserire quelle recensioni in una rete densa, le convertiamo in un vettore "multi-hot" di 10.000 posizioni (una per ogni parola del vocabolario): la posizione vale 1.0 se la parola appare nella recensione, 0.0 se non appare. Usiamo un sottoinsieme dei dati perché l'addestramento sia rapido durante la lezione.

In [ ]:
NUM_WORDS = 10000
N_TRAIN = 10000
N_VAL = 5000

(train_data, train_labels), (test_data, test_labels) = tf.keras.datasets.imdb.load_data(num_words=NUM_WORDS)

def multi_hot_sequences(sequences, dimension):
    """Convierte cada reseña (lista de IDs de palabra) en un vector de
    'dimension' posiciones: 1.0 si la palabra aparece, 0.0 si no aparece."""
    multi_hot = np.zeros((len(sequences), dimension), dtype="float32")
    for i, word_indices in enumerate(sequences):
        multi_hot[i, word_indices] = 1.0
    return multi_hot

x_train = multi_hot_sequences(train_data[:N_TRAIN], NUM_WORDS)
y_train = np.asarray(train_labels[:N_TRAIN]).astype("float32")

x_val = multi_hot_sequences(test_data[:N_VAL], NUM_WORDS)
y_val = np.asarray(test_labels[:N_VAL]).astype("float32")

print("Forma de x_train:", x_train.shape)
print("Forma de x_val:", x_val.shape)

# Así se ve una reseña ya codificada: casi todo ceros, con algunos unos
# en las posiciones de las palabras (de baja ID, las más frecuentes) que contiene
plt.figure(figsize=(9, 2))
plt.plot(x_train[0])
plt.xlabel("ID de palabra")
plt.ylabel("¿Aparece en la reseña?")
plt.title("Codificación multi-hot de la primera reseña de entrenamiento")
plt.show()

### Prepariamo i dati

Il dataset IMDB contiene 50 000 recensioni di film etichettate come positive o negative. Ogni recensione arriva già convertita in una lista di numeri: ogni numero è l'ID di una parola (l'ID 1 è la parola più frequente, l'ID 2 la seconda più frequente, e così via).

Per poter inserire quelle recensioni in una rete densa, le convertiamo in un vettore "multi-hot" di 10 000 posizioni (una per ogni parola del vocabolario): la posizione vale 1.0 se la parola appare nella recensione, 0.0 se non appare. Usiamo un sottoinsieme dei dati affinché l'addestramento sia veloce durante la lezione.

In [ ]:
def build_baseline_model():
    return tf.keras.Sequential([
        layers.Dense(16, activation="relu", input_shape=(NUM_WORDS,)),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])

def build_small_model():
    return tf.keras.Sequential([
        layers.Dense(4, activation="relu", input_shape=(NUM_WORDS,)),
        layers.Dense(4, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])

def build_big_model():
    return tf.keras.Sequential([
        layers.Dense(512, activation="relu", input_shape=(NUM_WORDS,)),
        layers.Dense(512, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])


def compile_and_fit(model, max_epochs=20, extra_callbacks=None):
    """Compila y entrena un modelo con early stopping incluido: si la
    pérdida de validación no mejora durante 5 épocas seguidas (patience=5),
    detiene el entrenamiento y recupera los mejores pesos vistos."""
    model.compile(optimizer="adam",
                  loss=tf.keras.losses.BinaryCrossentropy(),
                  metrics=["accuracy"])

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True)

    callbacks = [early_stop] + (extra_callbacks or [])

    history = model.fit(x_train, y_train,
                         epochs=max_epochs,
                         batch_size=512,
                         validation_data=(x_val, y_val),
                         callbacks=callbacks,
                         verbose=0)
    return history

In [ ]:
histories = {}
histories["baseline"] = compile_and_fit(build_baseline_model())
histories["small"] = compile_and_fit(build_small_model())
histories["big"] = compile_and_fit(build_big_model())

for name, history in histories.items():
    print(f"{name}: entrenó {len(history.epoch)} épocas (máximo 20, gracias a early stopping)")

In [ ]:
def plot_histories(histories_dict, metric="loss"):
    plt.figure(figsize=(9, 6))
    for name, history in histories_dict.items():
        epochs = range(1, len(history.history[metric]) + 1)
        linea = plt.plot(epochs, history.history[metric],
                          label=f"{name} (entrenamiento)")
        plt.plot(epochs, history.history["val_" + metric], "--",
                  color=linea[0].get_color(), label=f"{name} (validación)")
    plt.xlabel("Época")
    plt.ylabel(metric)
    plt.legend()
    plt.show()

plot_histories(histories)

Nota lo schema:

- **small** impiega più tempo a far scendere la sua perdita di addestramento: fa fatica ad apprendere (leggero underfitting)
- **baseline** fa scendere bene entrambe le curve e si mantengono ragionevolmente vicine
- **big** fa scendere molto rapidamente la sua perdita di addestramento, ma la sua perdita di validazione si separa presto e inizia a salire: overfitting evidente

## 2. Regolarizzazione: L2 e dropout

Prendiamo il modello **big**, che è quello che generalizza peggio, e applichiamo due tecniche di regolarizzazione contemporaneamente:

- **L2** (`kernel_regularizer=regularizers.l2(0.001)`): penalizza i pesi grandi nella funzione di perdita
- **Dropout** (`layers.Dropout(0.5)`): disattiva a caso il 50% dei neuroni dello strato precedente a ogni passo di addestramento

In [ ]:
def build_regularized_model():
    return tf.keras.Sequential([
        layers.Dense(512, activation="relu",
                     kernel_regularizer=regularizers.l2(0.001),
                     input_shape=(NUM_WORDS,)),
        layers.Dropout(0.5),
        layers.Dense(512, activation="relu",
                     kernel_regularizer=regularizers.l2(0.001)),
        layers.Dropout(0.5),
        layers.Dense(1, activation="sigmoid"),
    ])

histories["big_regularizado"] = compile_and_fit(build_regularized_model())

plot_histories({"big": histories["big"], "big_regularizado": histories["big_regularizado"]})

### Tre modelli, tre capacità diverse

- **baseline:** due livelli densi di 16 neuroni — una dimensione ragionevole
- **small:** due livelli densi di quattro neuroni — manca di capacità
- **big:** due livelli densi di 512 neuroni — ha troppa capacità

Tutti e tre terminano in un neurone con attivazione sigmoide, perché è una classificazione binaria (recensione positiva o negativa).

In [ ]:
# Entrenamos un modelo "baseline" de nuevo, esta vez guardando sus pesos
# en un checkpoint cada vez que mejora la pérdida de validación
modelo = build_baseline_model()
modelo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

checkpoint_path = "checkpoint_pesos.weights.h5"
cp_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    save_weights_only=True,
    save_best_only=True,
    monitor="val_loss",
    verbose=0,
)

modelo.fit(x_train, y_train, epochs=10, batch_size=512,
           validation_data=(x_val, y_val),
           callbacks=[cp_callback], verbose=0)

loss, acc = modelo.evaluate(x_val, y_val, verbose=0)
print(f"Precisión del modelo recién entrenado: {acc:.2%}")

In [ ]:
# Reconstruimos la MISMA arquitectura sin entrenar: su precisión debería
# rondar el 50% (equivalente a lanzar una moneda al aire)
modelo_sin_entrenar = build_baseline_model()
modelo_sin_entrenar.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

loss, acc = modelo_sin_entrenar.evaluate(x_val, y_val, verbose=0)
print(f"Precisión sin entrenar: {acc:.2%}")

# Cargamos en esa misma arquitectura los pesos que guardamos antes
modelo_sin_entrenar.load_weights(checkpoint_path)

loss, acc = modelo_sin_entrenar.evaluate(x_val, y_val, verbose=0)
print(f"Precisión tras cargar los pesos guardados: {acc:.2%}")

La precisione coincide (o quasi) con quella del modello originale: abbiamo recuperato la conoscenza appresa senza riaddestrare. Questo funziona solo perché `modelo_sin_entrenar` ha esattamente la stessa architettura del modello di cui abbiamo salvato i pesi.

### Salvare il modello completo

Se invece dei pesi vuoi salvare anche l'architettura e la configurazione dell'ottimizzatore, usa `model.save()` con il formato `.keras`, quello raccomandato in TensorFlow 2.x.

In [ ]:
modelo.save("modelo_clase04.keras")

# Cargamos el modelo completo en una variable nueva, sin necesidad de
# volver a definir la arquitectura en código
modelo_cargado = tf.keras.models.load_model("modelo_clase04.keras")
modelo_cargado.summary()

loss, acc = modelo_cargado.evaluate(x_val, y_val, verbose=0)
print(f"Precisión del modelo cargado desde 'modelo_clase04.keras': {acc:.2%}")

## Riepilogo

- Confronta la perdita di addestramento e di validazione a ogni epoca per rilevare overfitting (si separano) o underfitting (entrambe restano alte)
- Dropout, regolarizzazione L2 ed early stopping sono le tecniche più dirette contro l'overfitting, e combinarle di solito funziona meglio che usarne una sola
- `ModelCheckpoint` + `save_weights` / `load_weights` salvano solo i pesi: devi ricostruire la stessa architettura per riutilizzarli
- `model.save("archivo.keras")` + `tf.keras.models.load_model(...)` salvano e recuperano il modello completo, pronto per l'uso

Continua a esercitarti nel notebook `clase04_practica_buenas_practicas_entrenamiento.ipynb` e consulta il capitolo 06 del manuale per rivedere la teoria con più dettaglio.